In [ ]:
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from tqdm import tqdm
from pathlib import Path

Path("figures").mkdir(exist_ok=True)
Path("outputs").mkdir(exist_ok=True)

df = pd.read_parquet("data/news_with_topics_labeled.parquet")

model_path = "models/sentiment_distilbert_final"

tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSequenceClassification.from_pretrained(model_path)

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
model.eval()

In [ ]:
label_map = {
    0: "negative",
    1: "neutral",
    2: "positive"
}

score_map = {
    "negative": -1,
    "neutral": 0,
    "positive": 1
}

def predict_sentiment(text):
    inputs = tokenizer(
        text[:2000],
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=256
    ).to(device)
    
    with torch.no_grad():
        outputs = model(**inputs)
        probs = torch.softmax(outputs.logits, dim=1).cpu().numpy()[0]
    
    pred_id = probs.argmax()
    label = label_map[pred_id]
    
    return label, score_map[label], probs.max()

In [ ]:
sentiments = []

for text in tqdm(df["article_text"].tolist()):
    label, score, confidence = predict_sentiment(text)
    sentiments.append({
        "sentiment": label,
        "sentiment_score": score,
        "sentiment_confidence": confidence
    })

sent_df = pd.DataFrame(sentiments)

df = pd.concat([df.reset_index(drop=True), sent_df], axis=1)

df.to_parquet("data/news_topics_sentiment.parquet", index=False)

Topic-level sentiment

In [ ]:
topic_sentiment = (
    df.groupby("topic_label")
    .agg(
        articles=("article_text", "count"),
        avg_sentiment=("sentiment_score", "mean"),
        positive_share=("sentiment", lambda x: (x == "positive").mean()),
        negative_share=("sentiment", lambda x: (x == "negative").mean())
    )
    .reset_index()
    .sort_values("avg_sentiment", ascending=False)
)

topic_sentiment.to_csv("outputs/topic_sentiment.csv", index=False)

topic_sentiment

In [ ]:
import matplotlib.pyplot as plt

plot_df = topic_sentiment[topic_sentiment["articles"] >= 100].copy()
plot_df = plot_df.sort_values("avg_sentiment")

plt.figure(figsize=(10,7))
plt.barh(plot_df["topic_label"], plot_df["avg_sentiment"])
plt.axvline(0, linestyle="--")
plt.xlabel("Average Sentiment Score")
plt.title("Topic-Level Sentiment Toward AI Impact")
plt.tight_layout()
plt.savefig("figures/topic_sentiment.png", dpi=300)
plt.show()

Entity-level sentiment

In [ ]:
entities = pd.read_csv("outputs/entities_all.csv")

df_reset = df.reset_index().rename(columns={"index": "article_id"})

entity_sentiment = entities.merge(
    df_reset[["article_id", "sentiment", "sentiment_score"]],
    on="article_id",
    how="left"
)

entity_summary = (
    entity_sentiment.groupby("entity_clean")
    .agg(
        mentions=("article_id", "count"),
        avg_sentiment=("sentiment_score", "mean"),
        positive_share=("sentiment", lambda x: (x == "positive").mean()),
        negative_share=("sentiment", lambda x: (x == "negative").mean())
    )
    .reset_index()
)

entity_summary = entity_summary[entity_summary["mentions"] >= 30]
entity_summary = entity_summary.sort_values("mentions", ascending=False)

entity_summary.to_csv("outputs/entity_sentiment.csv", index=False)

entity_summary.head(30)

In [ ]:
top_entity_sentiment = entity_summary.sort_values("mentions", ascending=False).head(20)
top_entity_sentiment = top_entity_sentiment.sort_values("avg_sentiment")

plt.figure(figsize=(10,7))
plt.barh(top_entity_sentiment["entity_clean"], top_entity_sentiment["avg_sentiment"])
plt.axvline(0, linestyle="--")
plt.xlabel("Average Sentiment Score")
plt.title("Entity-Level Sentiment for Most Mentioned Organizations / Technologies")
plt.tight_layout()
plt.savefig("figures/entity_sentiment.png", dpi=300)
plt.show()

Sentiment over time

In [ ]:
df.columns

In [ ]:
DATE_COL = "date"

In [ ]:
df[DATE_COL] = pd.to_datetime(df[DATE_COL], errors="coerce")

df_time = df.dropna(subset=[DATE_COL]).copy()
df_time["month"] = df_time[DATE_COL].dt.to_period("M").dt.to_timestamp()

In [ ]:
monthly_sentiment = (
    df_time.groupby("month")
    .agg(
        articles=("article_text", "count"),
        avg_sentiment=("sentiment_score", "mean")
    )
    .reset_index()
)

monthly_sentiment.to_csv("outputs/monthly_sentiment.csv", index=False)

plt.figure(figsize=(11,5))
plt.plot(monthly_sentiment["month"], monthly_sentiment["avg_sentiment"], marker="o")
plt.axhline(0, linestyle="--")
plt.ylabel("Average Sentiment Score")
plt.xlabel("Month")
plt.title("AI News Sentiment Over Time")
plt.tight_layout()
plt.savefig("figures/sentiment_over_time.png", dpi=300)
plt.show()

In [ ]:
top_topic_labels = (
    df_time["topic_label"]
    .value_counts()
    .head(5)
    .index
)

topic_time = (
    df_time[df_time["topic_label"].isin(top_topic_labels)]
    .groupby(["month", "topic_label"])
    .agg(avg_sentiment=("sentiment_score", "mean"))
    .reset_index()
)

plt.figure(figsize=(12,6))

for topic in top_topic_labels:
    tmp = topic_time[topic_time["topic_label"] == topic]
    plt.plot(tmp["month"], tmp["avg_sentiment"], marker="o", label=topic)

plt.axhline(0, linestyle="--")
plt.ylabel("Average Sentiment Score")
plt.xlabel("Month")
plt.title("Topic-Level AI Sentiment Over Time")
plt.legend()
plt.tight_layout()
plt.savefig("figures/topic_sentiment_over_time.png", dpi=300)
plt.show()